In [2]:
%matplotlib tk
import torch
import os
import scipy as sc
import numpy as np
import sympy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import scienceplots
import matplotlib.animation as animation

plt.style.use(['science','notebook', 'grid'])
device = torch.device('cuda' if torch.cuda.is_available( ) else 'cpu')

In [3]:
def find_close_pairs_xy(xy, cutoff):
    """
    Efficiently find all pairs of points within 'cutoff' distance using spatial hashing.
    xy: numpy array of shape (2, N) where xy[0] = x, xy[1] = y
    cutoff: distance threshold
    Returns: numpy array of shape (M, 2) with (i, j) index pairs where distance < cutoff
    """
    x, y = xy[0], xy[1]
    cell_size = cutoff
    ix = np.floor(x / cell_size).astype(int)
    iy = np.floor(y / cell_size).astype(int)
    grid = {}
    for idx, (gx, gy) in enumerate(zip(ix, iy)):
        key = (gx, gy)
        if key not in grid:
            grid[key] = []
        grid[key].append(idx)
    neighbor_offsets = [(dx, dy) for dx in [-1,0,1] for dy in [-1,0,1]]
    pairs = set()
    for key, indices in grid.items():
        for dx, dy in neighbor_offsets:
            neighbor_key = (key[0]+dx, key[1]+dy)
            if neighbor_key in grid:
                for i in indices:
                    for j in grid[neighbor_key]:
                        if i < j:
                            dist = np.hypot(x[i]-x[j], y[i]-y[j])
                            if dist < cutoff:
                                pairs.add((i, j))
    if pairs:
        return np.array(list(pairs), dtype=int)
    else:
        return np.empty((0,2), dtype=int)

In [164]:
# Adaptive time stepping version of motion

def motion_adaptive(r, v, masses, ts, dt_init, d_cutoff, N):
    rs = torch.zeros((ts, r.shape[0], r.shape[1]), device=r.device)
    vs = torch.zeros((ts, v.shape[0], v.shape[1]), device=v.device)
    dts = torch.zeros(ts)
    rs[0] = r
    vs[0] = v
    dt = dt_init
    epsilon = 0.05  # Depth of the potential well
    sigma = 0.01   # Finite distance at which the inter-particle potential is zero
    drag_coeff = 0.5  # Drag coefficient (adjust as needed)
    for i in range(1, ts):
        print(f"Percent Complete: {(i+1) * 100 / ts:0.2f}%", end='\r')
        # Compute Van der Waals (Lennard-Jones) forces only for close pairs
        ax = torch.zeros(r.shape[1], device=r.device)
        ay = torch.zeros(r.shape[1], device=r.device)
        xy = r.cpu().numpy()
        close_pairs = find_close_pairs_xy(xy, cutoff=2*sigma)
        if close_pairs.shape[0] > 0:
            for idx in range(close_pairs.shape[0]):
                i1, i2 = close_pairs[idx]
                rij = r[:, i1] - r[:, i2]
                dist = torch.norm(rij)
                if dist > 1.123 * sigma:  # 1.1224620483 is approximately 2**(1/6)
                    # Lennard-Jones force
                    F_mag = 24 * epsilon * (2 * (sigma/dist)**12 - (sigma/dist)**6) / dist**2
                else:
                    F_mag = 7*epsilon*(1 - dist**2/(1.123 * sigma)**2)
                F_vec = F_mag * rij
                # Update accelerations
                ax[i1] += F_vec[0]
                ay[i1] += F_vec[1]
                ax[i2] -= F_vec[0]
                ay[i2] -= F_vec[1]
        # Add drag force (opposes velocity)
        ax -= drag_coeff * v[0]
        ay -= drag_coeff * v[1]
        # Boundary conditions
        v[0,r[0]>1] = -torch.abs(v[0,r[0]>1])
        v[0,r[0]<0] = torch.abs(v[0,r[0]<0])
        v[1,r[1]>1] = -torch.abs(v[1,r[1]>1])
        v[1,r[1]<0] = torch.abs(v[1,r[1]<0])
        r = r + v*dt
        ay = ay# - 9.81  # Gravity acceleration
        # Update velocities
        r[0, :] = r[0, :] + 0.5*ax*dt**2
        v[0, :] = v[0, :] + ax*dt
        r[1, :] = r[1, :] + 0.5*ay*dt**2
        v[1, :] = v[1, :] + ay*dt
        rs[i] = r
        vs[i] = v
    return rs, vs


In [167]:
# Use adaptive time stepping for the main simulation
N = 300
dt_init = 1e-2
t_steps = 500
v0 = 0.0 # Reduced initial velocity for better stability
L = 1
mass = 0.85e-2
r = torch.rand((2,N)).to(device)  
ixr = r[0]>1
ixl = r[0]<=1 
ids = torch.arange(N)
ids_pairs = torch.combinations(ids,2).to(device)
v = torch.zeros((2,N)).to(device)
v[0] = -v0*(r[1]-0.5)
v[1] = v0*(r[0]-0.5)
masses = mass*torch.ones(N).to(device)
radius = 0.005
rs, vs = motion_adaptive(r, v, masses, ts=t_steps, dt_init=dt_init, d_cutoff=4*radius, N=N)

In [168]:
fig, ax = plt.subplots(1,1,figsize=(5,5))
ax.clear()
vmin = 0
vmax = 1
ax.set_xlim(0,vmax)
ax.set_ylim(0,vmax)
markersize = 2 * radius * ax.get_window_extent().width  / (vmax-vmin) * 72./fig.dpi
red, = ax.plot([], [], 'o', color='red', markersize=markersize)
blue, = ax.plot([], [], 'o', color='blue', markersize=markersize)

# Only transfer to CPU for plotting, keep all other operations on GPU
def animate(i):
    # rs is already on GPU; only transfer the minimal data needed for plotting
    xred = rs[i][0][ixr].detach().cpu().numpy()
    yred = rs[i][1][ixr].detach().cpu().numpy()
    xblue = rs[i][0][ixl].detach().cpu().numpy()
    yblue = rs[i][1][ixl].detach().cpu().numpy()
    red.set_data(xred, yred)
    blue.set_data(xblue, yblue)
    return red, blue

writer = animation.FFMpegWriter(fps=30)
ani = animation.FuncAnimation(fig, animate, frames=t_steps, interval=50, blit=True)
#ani.save(filename="/Users/hasan/Python Animations/Self Gas Gravity.gif", writer="pillow")

In [70]:
x = np.linspace(0.056123, 0.07, 100)
y = 24 * (2 * (0.05/(x+1e-6))**12 - (0.05/(x+1e-6))**6) 
plt.plot(x, y)